# OWL (Web Ontology Language)

**Domain:** Symbolic AI & Logic  ·  **from study list**  ·  **runnable:** yes  ·  _owlready2_

A self-contained refresher on OWL — the W3C standard for **formal ontologies**: defining
classes, properties, and logical axioms rich enough that an automated **reasoner** can infer
new facts and check consistency.

## 1. What & Why

**OWL** (Web Ontology Language) is a W3C standard for writing **ontologies** — formal,
machine-readable models of a domain's concepts and the logical relationships between them.
It sits on top of RDF (see [`rdflib`](rdflib.ipynb)) and adds *expressive logical axioms*:
not just "rex is a Dog," but "every Dog is an Animal," "a Parent is anyone with at least one
child," "no one is both Alive and Dead." OWL's formal semantics come from **description logic**
(see [`description-logic-reasoners.ipynb`](description-logic-reasoners.ipynb)), so the meaning
of an ontology is unambiguous and a **reasoner** can compute logical consequences.

**The problem it solves.** A plain database or RDF graph stores facts but doesn't *understand*
them. OWL lets you encode the rules of a domain once, then a reasoner does three jobs for free:

- **Classification** — work out the full subclass hierarchy, including subsumptions you never
  stated explicitly (if `Pizza ⊓ hasTopping.Cheese ⊑ CheesyPizza`, it finds every cheesy pizza).
- **Realization** — infer which classes each individual belongs to from its properties.
- **Consistency / satisfiability checking** — flag contradictions and classes that can never
  have members, catching modeling bugs.

**Reach for it when** you have a genuinely *conceptual* domain — biomedical terminologies
(SNOMED CT, Gene Ontology), product catalogs, configuration validation, data integration across
schemas — and you want shared, vendor-neutral semantics plus inference. **Skip it when** you
just need to *store and query* graph data (use plain RDF + SPARQL), need *recursive rules over
data* (use [Datalog](datalog.ipynb) / [Prolog](swi-prolog.ipynb)), or need *arithmetic and
constraints* (use an [SMT solver](z3-smt.ipynb) or [MiniZinc](minizinc.ipynb)). OWL is about
*terminological* knowledge, not number crunching.

## 2. Mental Model

Think of OWL as a **schema that reasons about itself**.

A relational schema is dumb: it constrains shape but draws no conclusions. An OWL ontology is a
schema written as **logical statements**, so a reasoner can *deduce* the consequences:

```
You assert:                          The reasoner concludes:
  Dog ⊑ Pet ⊑ Animal                   every Dog is an Animal       (transitive subclass)
  Parent ≡ Person ⊓ ∃hasChild.Person   john hasChild mary  ⇒  john is a Parent  (realization)
  hasParent ≡ hasChild⁻               mary hasParent john          (inverse property)
  Cat ⊓ Dog ⊑ ⊥  (disjoint)            "x is a Cat and a Dog"  ⇒  inconsistent!
```

Two mantras that explain almost every surprising OWL result:

- **Open World Assumption (OWA).** What isn't stated is *unknown*, not false. A database says
  "no row ⇒ false"; OWL says "no axiom ⇒ could be either." You must *explicitly* close the world
  (disjointness, cardinality, `closed`/`only`) before the reasoner will conclude a negative.
- **No Unique Name Assumption (no-UNA).** Two different names may denote the *same* thing unless
  you say otherwise (`AllDifferent` / `differentFrom`). `:Bob` and `:Robert` could be one person.

Hold those two ideas and OWL's "why didn't it infer X?" moments mostly dissolve.

## 3. Key Concepts

| Term | What it is |
|---|---|
| **Individual** | A concrete object (an RDF instance): `rex`, `mary`. |
| **Class** | A *set* of individuals: `Dog`, `Person`. Classes can be built with logic. |
| **Property** | A binary relation. **Object** properties link individuals (`hasChild`); **data** properties link an individual to a literal (`age`). |
| **Axiom** | A logical statement: `Dog ⊑ Animal` (subclass), `Cat ⊓ Dog ⊑ ⊥` (disjoint), `hasParent ≡ hasChild⁻`. |
| **Restriction** | A class defined by a property constraint: `∃hasChild.Person` (*some* child is a Person), `∀hasPet.Cat` (*only* cats), `=2 hasParent` (cardinality), `∋ value`. |
| **Defined vs primitive class** | `≡` (equivalent_to, *necessary & sufficient*) lets the reasoner classify members **into** the class; `⊑` (subclass, necessary only) does not. |
| **TBox / ABox** | TBox = the *terminology* (class & property axioms). ABox = the *assertions* (individuals and their facts). |
| **Reasoner** | An engine (HermiT, Pellet, ELK, FaCT++) that computes the inferred hierarchy, realizes individuals, and checks consistency. |
| **OWL profiles** | Sublanguages trading expressivity for speed: **EL** (large terminologies, polynomial), **QL** (query rewriting over DBs), **RL** (rule engines). Full OWL-DL is decidable but worst-case expensive. |
| **Property characteristics** | `Transitive`, `Symmetric`, `Functional` (≤1 value), `InverseFunctional`, `Reflexive` — each licenses extra inferences. |

The single most important distinction is **`≡` vs `⊑`**: only *equivalent* (defined) classes pull
individuals in automatically. Beginners write `⊑` everywhere and wonder why nothing classifies.

## 4. Setup

We use **[owlready2](https://owlready2.readthedocs.io/)** — a Python library that maps OWL
ontologies onto Python classes and objects, and ships a bundled **HermiT/Pellet** reasoner.

```bash
pip install owlready2
```

Building, editing, and querying ontologies works with **pure Python, no extra dependencies**.
The *reasoner* (`sync_reasoner`) is a Java program, so it needs a **JRE on your PATH** — we gate
that cell behind a Java check so the notebook still runs top-to-bottom without Java installed.

In [1]:
# %pip install owlready2
import shutil, subprocess
from importlib.metadata import version

print("owlready2", version("owlready2"))


def java_available() -> bool:
    """The bundled reasoner is a Java program; check for a working JRE."""
    if not shutil.which("java"):
        return False
    try:
        return subprocess.run(
            ["java", "-version"], capture_output=True
        ).returncode == 0
    except Exception:
        return False


print("Java runtime available (reasoner can run):", java_available())

owlready2 0.51
Java runtime available (reasoner can run): False


## 5. Worked Examples

### Example 1 — Build a tiny ontology and read its structure

We define classes, an object property, a data property, and an individual entirely in Python.
owlready2 maps OWL classes to Python classes, so `issubclass` and instance creation just work.
This is the **TBox + ABox** with *no reasoner needed* — everything here is asserted or follows
from plain subclass transitivity.

In [2]:
from owlready2 import get_ontology, Thing, ObjectProperty, DataProperty

onto = get_ontology("http://example.org/zoo.owl")

with onto:                       # everything defined here belongs to `onto`
    class Animal(Thing): pass
    class Pet(Animal): pass      # Pet ⊑ Animal
    class Dog(Pet): pass         # Dog ⊑ Pet ⊑ Animal
    class Cat(Pet): pass
    class Human(Thing): pass

    class has_owner(ObjectProperty):     # individual -> individual
        domain = [Pet]
        range  = [Human]

    class call_name(DataProperty):       # individual -> literal
        range = [str]

# --- ABox: assert some individuals -------------------------------------
rex  = Dog("rex")
rex.call_name = ["Rex"]
alice = Human("alice")            # an owner
rex.has_owner = [alice]

print("Classes in ontology :", list(onto.classes()))
print("Dog ancestors       :", sorted(c.name for c in Dog.ancestors() if c is not Thing))
print("issubclass(Dog,Animal):", issubclass(Dog, Animal))
print("Subclasses of Pet   :", list(Pet.subclasses()))
print("rex is a            :", rex.is_a)
print("rex.call_name       :", rex.call_name)
print("rex.has_owner       :", rex.has_owner)

Classes in ontology : [zoo.Animal, zoo.Pet, zoo.Dog, zoo.Cat, zoo.Human]
Dog ancestors       : ['Animal', 'Dog', 'Pet']
issubclass(Dog,Animal): True
Subclasses of Pet   : [zoo.Dog, zoo.Cat]
rex is a            : [zoo.Dog]
rex.call_name       : ['Rex']
rex.has_owner       : [zoo.alice]


### Example 2 — A *defined* class + the reasoner (classification & realization)

Here is OWL's superpower. We declare `Parent` as a **defined class** (`equivalent_to`, i.e.
necessary *and* sufficient): *a Parent is any Person with at least one child who is a Person*.
We assert only that `john has_child mary` — we **never** say John is a Parent. We also make
`Male` and `Female` **disjoint**.

Running the reasoner should **realize** that `john` is a `Parent` and **classify** the hierarchy.
The reasoner cell is gated behind the Java check; without Java we print the asserted state and
the exact call you'd make, so the notebook still executes end-to-end.

In [3]:
from owlready2 import get_ontology, Thing, ObjectProperty, AllDisjoint

fam = get_ontology("http://example.org/family.owl")

with fam:
    class Person(Thing): pass
    class Male(Person): pass
    class Female(Person): pass
    AllDisjoint([Male, Female])          # a Male is never a Female

    class has_child(ObjectProperty):
        domain, range = [Person], [Person]

    class has_parent(ObjectProperty):
        inverse_property = has_child       # has_parent ≡ has_child⁻

    # DEFINED class: necessary AND sufficient → reasoner can classify INTO it.
    class Parent(Person):
        equivalent_to = [Person & has_child.some(Person)]

john = Person("john")
mary = Female("mary")
john.has_child = [mary]

print("Parent is defined as:", Parent.equivalent_to[0])
print("BEFORE reasoning -> john.is_a :", john.is_a)
# owlready2 keeps *declared* inverse properties in sync in Python automatically,
# no reasoner required — so has_parent is already populated here:
print("BEFORE reasoning -> mary.has_parent (auto inverse):", mary.has_parent)

Parent is defined as: family.Person & family.has_child.some(family.Person)
BEFORE reasoning -> john.is_a : [family.Person]
BEFORE reasoning -> mary.has_parent (auto inverse): [family.john]


In [4]:
from owlready2 import sync_reasoner_pellet

if java_available():
    with fam:
        # infer_property_values also materializes the inverse `has_parent`.
        sync_reasoner_pellet(fam, debug=0)
    print("AFTER reasoning -> john.is_a :", john.is_a)
    print("AFTER reasoning -> Parent instances:", list(Parent.instances()))
    print("(reasoner ran without raising -> ontology is consistent)")
else:
    print("No Java runtime found — skipping the reasoner.")
    print("With a JRE on PATH this cell would run:")
    print("    sync_reasoner_pellet(fam)")
    print("and the reasoner would conclude (via the defined class):")
    print("    john.is_a          -> [Person, Parent]   (realized)")
    print("    Parent.instances() -> [john]")

No Java runtime found — skipping the reasoner.
With a JRE on PATH this cell would run:
    sync_reasoner_pellet(fam)
and the reasoner would conclude (via the defined class):
    john.is_a          -> [Person, Parent]   (realized)
    Parent.instances() -> [john]


### Example 3 — Loading and saving real OWL files

OWL ontologies are exchanged as files (RDF/XML, Turtle, OWL/XML). owlready2 loads them with
`.load()` and serializes with `.save()`. Here we round-trip our family ontology to RDF/XML and
peek at the markup — the artifact you'd actually share, version, or feed to Protégé.

In [5]:
import os, tempfile

path = os.path.join(tempfile.gettempdir(), "family.owl")
fam.save(file=path, format="rdfxml")

size = os.path.getsize(path)
print(f"Saved {size} bytes to {path}\n")

with open(path) as fh:
    text = fh.read()
# Show the axiom that defines Parent (an owl:equivalentClass restriction).
for line in text.splitlines():
    if "Parent" in line or "Restriction" in line or "equivalentClass" in line:
        print(line.strip())

Saved 1772 bytes to /var/folders/p8/sm5jmh055md_zzhhn1mfgyw80000gn/T/family.owl

<owl:Class rdf:about="#Parent">
<owl:equivalentClass>
<owl:Restriction>
</owl:Restriction>
</owl:equivalentClass>


## 6. Gotchas & Pitfalls

- **`subclass_of` (⊑) won't classify individuals; only `equivalent_to` (≡) does.** This is the
  #1 beginner trap. A *primitive* subclass is a one-way "if member then…"; you need a *defined*
  class (necessary **and** sufficient) before the reasoner pulls individuals in.
- **Open World Assumption bites constantly.** The reasoner won't conclude "John has no other
  children," "this is the only topping," or "X is not a Y" unless you *close* the world with
  disjointness, cardinality, or `only` (∀) restrictions. Missing data ≠ false.
- **No Unique Name Assumption.** Without `AllDifferent` / `differentFrom`, two individuals may be
  inferred *identical* — which can silently merge facts or trigger surprising cardinality results.
- **`some` vs `only` (∃ vs ∀).** `hasPet some Cat` means "has at least one cat"; `hasPet only Cat`
  means "has *no* non-cat pets" — and `only` is vacuously true for someone with **no** pets at all.
- **Reasoning can be slow or blow up.** Full OWL-DL is N2ExpTime-hard in the worst case. Deeply
  nested restrictions, many cardinalities, and large ABoxes hurt. If you only need a fragment,
  target an **EL/QL/RL profile** and a profile-specific reasoner (ELK is fast for EL).
- **owlready2 needs Java for `sync_reasoner`.** Pure construction/query is Python-only, but
  HermiT and Pellet are Java; "reasoner not found" almost always means no JRE on PATH.
- **`with onto:` matters.** Classes/properties defined outside an ontology block don't get
  attached where you expect; always define inside `with onto:`.
- **Punning & metamodeling.** OWL-DL keeps individuals, classes, and properties in separate
  layers. Treating a class as an individual ("OWL Full") makes reasoning undecidable.

## 7. When to Use vs Alternatives

| You want to… | Best tool | Why not OWL |
|---|---|---|
| Share a formal, vendor-neutral **domain model** with inference (taxonomies, biomedical, config validation) | **OWL** ✅ | — this is exactly its niche |
| Just **store + query** graph/triple data | Plain **RDF + SPARQL** ([rdflib](rdflib.ipynb), [SPARQL](sparql.ipynb)) | OWL's axioms are overkill; SPARQL queries, doesn't classify |
| **Recursive rules over facts** (graph reachability, policies) | [**Datalog**](datalog.ipynb) / [**Prolog**](swi-prolog.ipynb) | OWL's rule support (SWRL) is limited; CWA rules are simpler |
| **Arithmetic, scheduling, constraints** | [**SMT**](z3-smt.ipynb) / [**MiniZinc**](minizinc.ipynb) | OWL has essentially no arithmetic reasoning |
| Forward-chaining **production rules** | [**RETE / CLIPS**](rete-algorithm.ipynb) | Different paradigm: imperative rules, closed world |
| **Probabilistic** logic | [**ProbLog**](problog.ipynb) | OWL is strictly Boolean / monotonic |

**Honest trade-offs.** OWL's strengths are *standardization* (W3C, huge tooling: Protégé,
triple stores, SPARQL), *decidable* description-logic semantics, and *automatic classification*.
Its weaknesses are the OWA/no-UNA surprises, no native arithmetic, monotonic-only reasoning, a
real learning curve, and reasoner cost at scale. Many "knowledge graph" projects use OWL/RDFS
*lightly* (a vocabulary) and lean on SPARQL + property graphs for the heavy lifting — reserve
full OWL-DL reasoning for domains where formal classification genuinely earns its keep.

## 8. Resources

- **OWL 2 Primer** (the gentle official intro) — <https://www.w3.org/TR/owl2-primer/>
- **OWL 2 Structural Specification** (the normative reference) — <https://www.w3.org/TR/owl2-syntax/>
- **owlready2 documentation** (this notebook's library) — <https://owlready2.readthedocs.io/>
- **Protégé** (the standard free OWL editor + reasoner GUI) — <https://protege.stanford.edu/>
- **A Practical Guide To Building OWL Ontologies (Pizza Tutorial)** — <https://www.michaeldebellis.com/post/new-protege-pizza-tutorial>
- **OWL 2 Profiles** (EL/QL/RL, when to use which) — <https://www.w3.org/TR/owl2-profiles/>

**Related notebooks:** [`rdflib`](rdflib.ipynb) (the RDF foundation), [`sparql`](sparql.ipynb)
(querying), [`description-logic-reasoners`](description-logic-reasoners.ipynb) (the logic under
the hood), [`knowledge-graphs`](knowledge-graphs.ipynb) (the broader picture).